# Single-particle orbit in a periodic electrostatic potential

This experiment evolves one full-cyclotron particle with classical RK4. The animation shows the particle, its accumulated orbit, and the time-dependent periodic potential.

All potential, physical, initial-condition, and numerical parameters are explicit below so that the result can be reproduced.

API migration: this notebook uses the current simulation API. Physical rho and eta belong to dynamics or study settings; initial configurations store geometry. Stored outputs were cleared and should be regenerated before scientific interpretation.


In [ ]:
%matplotlib inline

import numpy as np

from dynamics import FullCyclotronDynamics
from initial_conditions import FCInitialConfiguration
from simulation import (
    InitialValueProblem,
    RK4,
    SimulationRequest,
    simulate,
)
from studies import RandomPotentialConfig
from visualization import (
    animate_fc_particle_solution,
    display_animation,
)

## Reproducible configuration

`rho` sets the normalized Larmor radius and `eta` sets the signed cyclotron time scale. The initial velocity coordinates are normalized model coordinates; their physical position-rate scale is reported by `trajectory.velocity_scale`.

In [ ]:
potential_config = RandomPotentialConfig(
    amplitude=0.2,
    max_wave_number=8,
    nx=48,
    ny=48,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()

rho = 0.2
eta = 0.1
initial_position = (np.pi, np.pi)
initial_velocity = (1.0, 0.0)

t_span = (0.0, 8 * np.pi)
max_step = 0.002
output_sample_count = 501

trajectory = FCInitialConfiguration.from_components(
    x=np.asarray([initial_position[0]]),
    y=np.asarray([initial_position[1]]),
    vx=np.asarray([initial_velocity[0]]),
    vy=np.asarray([initial_velocity[1]]),
    
    
)
dynamics = FullCyclotronDynamics(potential, rho=rho, eta=eta)
problem = InitialValueProblem(dynamics, trajectory)
request = SimulationRequest.uniform(
    t_span=t_span,
    max_step=max_step,
    sample_count=output_sample_count,
)

print(potential_config)
print(f"Position-rate scale: {dynamics.velocity_scale:.3f}")

## Integrate the orbit

In [ ]:
solution = simulate(problem, RK4(), request)

print(f"Fixed RK4 steps: {solution.diagnostics['step_count']}")

## Animated orbit

The potential and particle state are shown at the same saved times. The black curve is the orbit accumulated up to the current frame.

In [ ]:
animation = animate_fc_particle_solution(
    potential,
    solution,
    frames=151,
    interval=80,
    cmap="RdBu_r",
    repeat=True,
)

display_animation(animation, embed_limit_mb=30.0)